In [1]:
import torch
import torch.nn as nn
import numpy as np

# GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Import and inspect the CodeGen model

In [2]:
# Model source:
# https://huggingface.co/Salesforce/codegen-350M-mono
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained('Salesforce/codegen-350M-mono')

model = AutoModelForCausalLM.from_pretrained('Salesforce/codegen-350M-mono').to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

In [3]:
model

CodeGenForCausalLM(
  (transformer): CodeGenModel(
    (wte): Embedding(51200, 1024)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-19): 20 x CodeGenBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): CodeGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (mlp): CodeGenMLP(
          (fc_in): Linear(in_features=1024, out_features=4096, bias=True)
          (fc_out): Linear(in_features=4096, out_features=1024, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=51200, bias=True)
)

In [4]:
# Analyzing tokenizer
print(f'Tokenizer has {tokenizer.vocab_size:,} tokens.\nA few random tokens:\n')

for i in range(30):
  # generate a random token
  randtok = torch.randint(tokenizer.vocab_size,(1,))
  print(f'Token {randtok[0]:5} is "{tokenizer.decode(randtok)}"')

Tokenizer has 50,257 tokens.
A few random tokens:

Token 48503 is " Baird"
Token  7012 is "ba"
Token  8820 is " uncom"
Token 16873 is "atible"
Token 24479 is " unbelievable"
Token 46489 is "Nich"
Token 22729 is " deficits"
Token 26604 is "¶"
Token 36627 is " fellows"
Token 36527 is " additionally"
Token 12230 is " Malays"
Token 21141 is "inski"
Token 10665 is " Affairs"
Token 31473 is " seafood"
Token 45208 is " freshmen"
Token 48902 is "Assistant"
Token 49232 is " Nationwide"
Token 16917 is " Brig"
Token 44616 is "ratom"
Token 49914 is "reporting"
Token 36145 is " pedd"
Token 32810 is "ahi"
Token 12033 is " Interest"
Token  6218 is " messages"
Token  6438 is " 裏"
Token 27936 is "315"
Token 45658 is "nda"
Token 29560 is " Opposition"
Token 46644 is " eclectic"
Token 15696 is " 106"


# Generate a response to some code input

In [9]:
text = 'for i in range(10):'
tokenized_input = tokenizer(text, return_tensors='pt')
input_ids = tokenized_input['input_ids'].to(device)
attention_mask = tokenized_input['attention_mask'].to(device)

generated_ids = model.generate(input_ids, attention_mask=attention_mask, max_length=222, temperature=1.4, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

for i in range(10):
    test_sample = [s[:9]]
    solver_info_numpy.train(problem, test_sample, solver=solver_type)[0][0]


def test_sorting_samples_to_satisfy_and_restore0():
    np.random.seed(12)
    problem_type = ProblemType.solving_eager_samples_in_one_problem
    problem = EqProblem()
    sorted_samples_n = 5

    sorting_list = [(np.random.uniform() * 99),
                    int((np.random.uniform()) + 0),
                    int((2 * np.random.uniform(size=10) - 1) * sorted_samples_n), int()]
    sorted_samples, unpermuted = npsorted.take(sorting_list, axis=None)

    sampler0 = SortSam


# Import git-calc files and tokenize

In [11]:
import os
import subprocess
import nbformat

# download the repo
repo_url = 'https://github.com/mikexcohen/Calculus_book.git'
repo_dir = 'Calculus_book'
subprocess.run(['git','clone',repo_url,repo_dir])

# initialize a list for all tokens
all_tokens = []


## find all .ipynb files
for root, dirs, files in os.walk(repo_dir):
  for file in files:
    if file.endswith('.ipynb'):

      # load and process notebooks
      # print(f'Processing notebook: {os.path.join(root, file)}')
      with open(os.path.join(root, file),'r',encoding='utf-8') as f:

        # parse the notebook structure
        nb_data = nbformat.read(f,as_version=4)

        # gather all code cells into a single string
        code_cells = []
        for cell in nb_data['cells']:
          if cell['cell_type'] == 'code': # this cell contains code
            code_cells.append(cell['source']) # the actual code in this cell

        # join code cells
        all_code = '\n'.join(code_cells)

        # tokenize and add to the list of all tokens
        all_tokens += tokenizer(all_code)['input_ids']

# check the final dataset size
print(f'\n\nTraining data contains {len(all_tokens):,} tokens, of which {len(set(all_tokens)):,} are unique.')

Token indices sequence length is longer than the specified maximum sequence length for this model (3450 > 2048). Running this sequence through the model will result in indexing errors




Training data contains 166,818 tokens, of which 3,133 are unique.


# Train the model

In [37]:
# Find the number of trainable parameters in the model
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'The model has {trainable_params:,} trainable parameters.')

The model has 18,874,368 trainable parameters.


In [35]:
# Modify paarmeters to train the 0-5 attention layers only
for name, param in model.named_parameters():
  param.requires_grad = False

for name, param in model.transformer.h[:6].named_parameters():
  if 'qkv' in name:
    param.requires_grad = True

In [15]:
!pip install torchinfo
from torchinfo import summary

In [36]:
# Assuming a typical max sequence length for CodeGen models
max_seq_len = 2048

# Prepare sample input_ids and attention_mask from a slice of all_tokens
sample_input_ids = torch.tensor(all_tokens[:max_seq_len]).unsqueeze(0).to(device)
sample_attention_mask = torch.ones(sample_input_ids.shape, dtype=torch.long).to(device)

# Pass the sample inputs as a dictionary to summary
summary(model,
        input_data={'input_ids': sample_input_ids, 'attention_mask': sample_attention_mask},
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
        )

Layer (type (var_name))                            Input Shape          Output Shape         Param #              Trainable
CodeGenForCausalLM (CodeGenForCausalLM)            --                   --                   --                   Partial
├─CodeGenModel (transformer)                       [1, 2048]            --                   --                   Partial
│    └─Embedding (wte)                             [1, 2048]            [1, 2048, 1024]      (52,428,800)         False
│    └─Dropout (drop)                              [1, 2048, 1024]      [1, 2048, 1024]      --                   --
│    └─ModuleList (h)                              --                   --                   --                   Partial
│    │    └─CodeGenBlock (0)                       [1, 2048, 1024]      [1, 2048, 1024]      12,590,080           Partial
│    │    └─CodeGenBlock (1)                       [1, 2048, 1024]      [1, 2048, 1024]      12,590,080           Partial
│    │    └─CodeGenBlock (2) 

In [42]:
numsteps = 300
batch_size = 32

seq_len = 256

train_loss = []

model.to(device)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=.01)

# Convert all_tokens to a tensor once before the training loop
all_tokens_tensor = torch.tensor(all_tokens, dtype=torch.long).to(device)

for step in range(numsteps):

  ix = torch.randint(0,len(all_tokens_tensor)-seq_len, size=(batch_size,)).to(device)
  X  = all_tokens_tensor[ix[:, None] + torch.arange(seq_len).to(device)]

  optimizer.zero_grad(set_to_none=True)

  out = model(X, labels=X)
  loss = out.loss
  loss.backward()
  optimizer.step()


  train_loss.append(loss)

  if step % 50 == 0:
    print(f'Step {step}: Loss = {loss:.4f}')

Step 0: Loss = 1.4293
Step 50: Loss = 1.0554
Step 100: Loss = 0.8500
Step 150: Loss = 0.7357
Step 200: Loss = 0.6617
Step 250: Loss = 0.6461


In [43]:
text = 'for i in range(10):'
tokenized_input = tokenizer(text, return_tensors='pt')
input_ids = tokenized_input['input_ids'].to(device)
attention_mask = tokenized_input['attention_mask'].to(device)

generated_ids = model.generate(input_ids, attention_mask=attention_mask, max_length=222, temperature=1.4, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

for i in range(10):
# i represents the ith complex fraction component
l_c   = uC_x * uCX

ax3.plot(x[np.logical_or(l_l,l_c)],l_l[np.logical_or(l_l,np.isfinite(l_l))],'ko',markersize=5,markerfacecolor='w',linewidth=2,zorder=-4)
ax3.plot(x[~l_l],x[~l_l],'ko',markerfacecolor='k',markersize=5,linewidth=1)
ax4.plot(theta,l,'-ok','k.',color='k',label='Linear approximation')
ax5.plot(theta,f(theta),'k':'-',color=[.8,.8,.8],label='Approximation error')
for t in thetas: ax5
